In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [5]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model="ollama:llama3.1:8b",
    tools=tools,
    system_prompt=prompt
)

In [9]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [10]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='247c6184-7c1c-415a-97d6-6239d77da985'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-09-03T08:09:34.5983476Z', 'done': True, 'done_reason': 'stop', 'total_duration': 16678732800, 'load_duration': 12203402800, 'prompt_eval_count': 297, 'prompt_eval_duration': 740927000, 'eval_count': 23, 'eval_duration': 3720260000, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--01a06650-ab5a-7041-aa2b-73a40e4d97e0-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': '5288d4c9-4218-4416-9fd5-00674c954ebd', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 297, 'output_tokens': 23, 'total_tokens': 320}),
              ToolMessage(content=[{'type': 'text', 'text': '{\n  "query": "langch

## Online MCP

In [11]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [12]:
agent = create_agent(
    model="ollama:llama3.1:8b",
    tools=tools,
)

In [13]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='56fce45b-3a36-40ed-9881-f56fa65ba4f4'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-09-03T08:14:40.2068241Z', 'done': True, 'done_reason': 'stop', 'total_duration': 17224486400, 'load_duration': 13001042900, 'prompt_eval_count': 362, 'prompt_eval_duration': 810528000, 'eval_count': 21, 'eval_duration': 3389186000, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--01a06655-5301-73a1-a506-0c4dc07a6be1-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/New_York'}, 'id': '26e234c0-7ca1-4545-bb16-ef6798e34cf8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 362, 'output_tokens': 21, 'total_tokens': 383}),
              ToolMessage(content=[{'type': 'text', 'text': '{\n  "timezone": "America/New_York",\n  "datetime": "2026-